In [21]:
from rag_retrieval import get_matching_chunks,generate_query_embedding
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please check your .env file.")

In [4]:
query = "What lounge access or travel insurance benefits do I get with the Platinum Travel card?"
chunks = get_matching_chunks(query)

Embedding user query: 'What lounge access or travel insurance benefits do I get with the Platinum Travel card?'...


In [27]:
print(chunks[0])

{'document_id': 'DOC_1783939125', 'card_name': 'Platinum Travel', 'issuer': 'AE', 'document_type': 'Terms and Conditions', 'page_number': 3, 'effective_date': '2026-07-13', 'source_url': 'https:/storage.googleapis.com/my_storage_files/.\\data\\raw_pdfs\\AE_platinum_travel.pdf', 'chunk_text': '1800 -419-2122, 0124-2801122. Airport Lounges • The benefit is available only to the Primary/Basic Cardmember of American Express Platinum Travel Credit Card • American Express Cardmembers can enjoy all the Card related benefits as long as their accounts (including all linked accounts) are in good standing • The benefits under this programme are being made by American Express / Partners of American Express on a best effort basis and are subject to availability and Cardmembers must exercise due diligence in understanding specific terms that may be applicable to such benefits. • Programme is open only for Cardmembers carrying a same day ticket for airline travel that allows clearance through securit

In [5]:


llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.0
)

In [11]:

def edit_query(query: str) -> str:
    prompt = f"""
Expand this user query for retrieval.
Add a few useful similar words.
Return only one short expanded query.

Query: {query}
"""
    return llm.invoke(prompt).content.strip()

def make_keywords(text, max_words=6):
    words = text.lower().replace("?", "").replace(",", "").replace(".", "").replace(')', '').replace('(', '').split()
    words = list(dict.fromkeys(words))   # remove duplicates, keep order
    # print(f"Keywords: {words[:max_words]}")
    return words

# query_text = "can you explain Fuel and Mileage policy?"
edit_query(query)
make_keywords(query)

['what',
 'lounge',
 'access',
 'or',
 'travel',
 'insurance',
 'benefits',
 'do',
 'i',
 'get',
 'with',
 'the',
 'platinum',
 'card']

In [12]:
def classify_intent(query: str, intent_list: list) -> str:
    """
    Classifies user intent into ONE label from intent_list.
    """

    system_prompt = (
        "You are an intent classification engine. "
        "You must classify the user's query into exactly ONE intent "
        "from the provided list. "
        "Return ONLY the intent label. Do not explain."
    )

    user_prompt = f"""
Intents:
{', '.join(intent_list)}

User Query:
"{query}"

Return only one intent from the list above.
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]

    response = llm.invoke(messages)

    return response.content.strip()

In [8]:
def render_prompt(query: str, sources: str) -> str:
  PROMPT_TEMPLATE = """
Imagine you're a financial assitant, your job is to give grounded answers using only the Context below:
Make sure you don't answer from outside the given context, don't be robotic, and please answer in a friendly way.

Context:
{sources}

Question:
{query}
""".strip()
  return PROMPT_TEMPLATE.format(query=query, sources=sources)

In [41]:
def build_context(results, max_chars: int = 1200) -> str:
    parts = []
    total_chars = 0

    for item in results:
        chunk = item["chunk_text"].strip()

        remaining = max_chars - total_chars
        if remaining <= 0:
            break

        if len(chunk) > remaining:
            chunk = chunk[:remaining]

        parts.append(chunk)
        total_chars += len(chunk)

    return "\n\n".join(parts)

In [15]:
# query = "can i reimburse fuel?"

query_type = ["Single transaction recommendation", "Monthly spend optimization", "Point transfer strategy", "Card comparison", "Missing information query"]

query = "I want compare my hdfc card with sbi card for fuel and mileage benefits"
intent = classify_intent(query, query_type)

print(intent)

Card comparison


In [37]:
def retrieve_docs(query_text, keywords=None, top_k=5,query_intent = intent):
    # query_vector = generate_query_embedding(query_text)

    # if keywords:
    #     results = index.query(
    #         vector=query_vector,
    #         top_k=top_k,
    #         include_metadata=True,
    #         filter={"keywords": {"$in": keywords}}
    #     )
    # else:
    results = get_matching_chunks(query_text)



    # response = []
    # for match in results["matches"]:
    #     response.append({
    #         "id": match["id"],
    #         "score": float(match["score"]),
    #         "text": match["metadata"]["text"]
    #         # "links":
    #     })
# Home work - Put links from metadata in the response
    return results

In [34]:
r_docs = retrieve_docs(query)
r_docs

Embedding user query: 'I want compare my hdfc card with sbi card for fuel and mileage benefits'...


[{'document_id': 'DOC_1783939177',
  'card_name': 'Cashback',
  'issuer': 'SBI',
  'document_type': 'Terms and Conditions',
  'page_number': 42,
  'effective_date': '2026-07-13',
  'source_url': 'https:/storage.googleapis.com/my_storage_files/.\\data\\raw_pdfs\\SBI_cashback.pdf',
  'chunk_text': 'in this clause and/or should be decided under the Arbitration and Conciliation Act, 1996. 10.3 Any arbitration award granted shall be final and binding on the Parties. The venue and seat of the Arbitral Tribunal shall be at New Delhi. 10.4 This Clause 10 shall survive termination of the Cardholder Agreement. 40 13. OTHER TERMS AND CONDITIONS – PRODUCT FEATURES 13.1 Terms and Conditions: Fuel Surcharge • CASHBACK SBI Card: 1% Fuel Surcharge waiver for each transaction between 500 and 3000. Maximum Surcharge waiver of 100 per statement cycle, per credit card account • Fuel Surcharge is applicable for transactions done for MCCs 5172, 5541, 5542 and 5983 13.2 Terms & Conditions for Card Upgrade • 

In [36]:
r_docs[0]['document_id']

'DOC_1783939177'

In [47]:
def _start(query):
  return {"query":query}

final_chain = (
    RunnableLambda(_start)
    .assign(edited_query=RunnableLambda(lambda d: edit_query(d["query"])))
    .assign(intent=RunnableLambda(lambda d: classify_intent(query=d["edited_query"], intent_list=query_type)))
    .assign(filter_keywords=RunnableLambda(lambda d: make_keywords(d["edited_query"], max_words=6)))
    .assign(results=RunnableLambda(lambda d: retrieve_docs(d["edited_query"], keywords=d["filter_keywords"] , query_intent=d["intent"], top_k=3)))
    .assign(sources = RunnableLambda(lambda d: build_context(d['results'], max_chars = 1200))) # {'sources':All sources combined}
    .assign(prompt = RunnableLambda(lambda d: render_prompt(d['edited_query'], d['sources']))) # {'prompt': 'updated prompt'}
    .assign(answer = RunnableLambda(lambda d: llm.invoke(d['prompt'])))
    .pick('answer') | StrOutputParser()
)

final_output = final_chain.invoke("“Based on the retrieved travel reward rules, Card A appears to give the highest reward for a ₹50,000 flighttransaction. The estimated reward value is ₹2,500, assuming each point is worth ₹1. Card B gives anestimated ₹1,500 value. Card C gives ₹1,000 cashback. Therefore, Card A is recommended, provided yourmonthly cap has not been exhausted.”?")
final_output

Embedding user query: 'Expanded Query: “Considering the analyzed travel reward policies, Card A seems to offer the maximum benefits for a ₹50,000 flight purchase, with an estimated reward value of ₹2,500, assuming each point equals ₹1. In comparison, Card B provides an estimated value of ₹1,500, while Card C offers ₹1,000 cashback. Thus, Card A is suggested, as long as your monthly limit has not been reached. Please include terms like 'travel rewards', 'credit card benefits', 'cashback offers', and 'monthly spending limits' for better retrieval.”'...


"It sounds like you're diving into the world of travel rewards and credit card benefits! Based on the context provided, if you're considering a ₹50,000 flight purchase, using the Axis Bank Atlas Credit Card could be a great option for maximizing your travel rewards. \n\nWith this card, you would earn 5 EDGE Miles for every ₹100 spent on eligible transactions, which includes direct airline purchases. For a ₹50,000 flight, that would translate to earning 2,500 EDGE Miles (since ₹50,000 is 50,000/100 = 500, and 500 x 5 = 2,500). If we assume each EDGE Mile is valued at ₹1, that gives you a reward value of ₹2,500, which aligns with your findings for Card A.\n\nJust keep in mind that the earning of 5 EDGE Miles is capped at ₹2 lakh in spending per month. So as long as you haven't reached that monthly spending limit, the Axis Bank Atlas Credit Card could indeed be the best choice for your travel rewards compared to Card B and Card C, which offer lower estimated values. \n\nIf you have any mo